In [1]:
import pandas as pd
from preprocessing.preprocessing_full import preprocessing_full
test_catedra = pd.read_csv("data/raw/SUVS_2025-test-masked.csv")

test_catedra = preprocessing_full(test_catedra, precio = False)
test_catedra.to_csv("data/numeric/test_catedra_numeric.csv", index=False)

[TOTAL] Muestras eliminadas en total: 0

✔️ Marcas imputadas: 0
🗑️ Muestras eliminadas por no poder imputar: 0

✔️ Modelos imputados: 0
🗑️ Muestras eliminadas por no poder imputar: 0

✔️ Muestras version train imputadas: 172
🗑️ Muestras eliminadas por no poder imputar: 3
   ➤ Versiones imputadas por contexto: 0
   ➤ Versiones corregidas por similitud: 1628
   ➤ Versiones imputadas por moda del modelo (último paso): 213

✔️ Imputaciones Combustible realizadas: 0
🗑️ Muestras eliminadas: 0 

✔️ Imputaciones directas: 127
📦 Fallback por modelo: 0
🌍 Fallback global: 4
✅ Total imputado: 131
🗑️ Muestras eliminadas: 0

✔️ Imputaciones por coincidencia directa: 999
📦 Fallback por modelo: 0
🌍 Fallback global: 119
✅ Total imputado: 1118
🗑️ Muestras eliminadas: 0

✔️ Imputaciones de motor por coincidencia directa: 14
📦 Fallback por modelo: 0
🌍 Fallback global: 6
✅ Total imputado: 20
🗑️ Muestras eliminadas: 0

✔️ Imputaciones de camara realizadas por coincidencia exacta: 3063
🔧 Muestras sin coincid

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

%reload_ext autoreload
%autoreload 2

train = pd.read_csv('data/numeric/train_numeric.csv')
val = pd.read_csv('data/numeric/val_numeric.csv')
test = pd.read_csv('data/numeric/test_catedra_numeric.csv')

In [3]:
try:
    import mlflow
except ImportError:
    !pip install mlflow


import mlflow.sklearn



/Users/kekozim/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


### BARRIDO DEL DATASET

In [4]:
#barrido del dataset train
pd.set_option('display.expand_frame_repr', False)
print(train.sample(5))

      Marca  Modelo     Año  Versión  Color  Tipo de combustible  Motor  tiene_camara  Kilómetros   Precio  Tipo de vendedor     HP  is_4x4  is_manual  is_coupe
2096     17      17  2024.0      155     11                    3    1.3             1         0.0  42400.0                 0  175.0       1          0         0
6160     10       4  2018.0       92      4                    3    1.4             1     98000.0  16800.0                 0  150.0       0          1         0
288      17      12  2008.0      153     19                    3    3.7             0    179000.0  15000.0                 0  205.0       0          0         0
2458     32      78  2025.0      335      2                    3    1.4             1         0.0  35368.0                 0  150.0       0          0         0
3630     25     107  2009.0      234     11                    3    2.5             1    182427.0   9000.0                 1  170.0       1          1         0


### Prueba regresion lineal simple


In [5]:
# Conectarse al servidor local de MLflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")

# Crear o usar experimento
mlflow.set_experiment("Predicción Precio SUVs")
with mlflow.start_run(run_name="Regresión Lineal Simple"):

    # División de datos
    x_train = train.copy().drop(columns=['Precio'])
    y_train = train['Precio']
    x_val = val.copy().drop(columns=['Precio'])
    y_val = val['Precio']

    # Escalado
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_val = scaler.transform(x_val)

    # Entrenar modelo
    modelo = LinearRegression()
    modelo.fit(x_train, y_train)

    # Predicción y métricas en entrenamiento
    y_pred_train = modelo.predict(x_train)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    r2_train = r2_score(y_train, y_pred_train)

    # Predicción y métricas en validación
    y_pred_val = modelo.predict(x_val)
    rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
    r2_val = r2_score(y_val, y_pred_val)

    # Log de métricas
    mlflow.log_metric("rmse_val", rmse_val)
    mlflow.log_metric("r2_val", r2_val)

    # Log de hiperparámetros
    mlflow.log_param("model_type", "LinearRegression")

    # Guardar modelo
    mlflow.sklearn.log_model(modelo, "modelo_lr")

    print(f"✅ RMSE validación: {rmse_val:.4f}")
    print(f"✅ R2 validación: {r2_val:.4f}")

2025/07/02 17:52:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/07/02 17:52:57 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✅ RMSE validación: 11081.6722
✅ R2 validación: 0.6003
🏃 View run Regresión Lineal Simple at: http://127.0.0.1:5000/#/experiments/454211106375854603/runs/dc25db461e454e6a8b9fe566a80a0b3a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/454211106375854603


In [6]:
from sklearn.linear_model import Ridge, Lasso
# Entrenar modelo Ridge
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(x_train, y_train)
# Predecir con validación
y_pred_ridge = ridge_model.predict(x_val)
# Comparar contra los valores reales de validación
rmse_ridge = np.sqrt(mean_squared_error(y_val, y_pred_ridge))
print(f"RMSE validación Ridge: {rmse_ridge:.4f}")
r2_val_ridge = r2_score(y_val, y_pred_ridge)
print(f"R2 validación Ridge: {r2_val_ridge:.4f}")
# Entrenar modelo Lasso
lasso_model = Lasso(alpha=0.1)
lasso_model.fit(x_train, y_train)
# Predecir con validación
y_pred_lasso = lasso_model.predict(x_val)
# Comparar contra los valores reales de validación
rmse_lasso = np.sqrt(mean_squared_error(y_val, y_pred_lasso))
print(f"RMSE validación Lasso: {rmse_lasso:.4f}")
r2_val_lasso = r2_score(y_val, y_pred_lasso)
print(f"R2 validación Lasso: {r2_val_lasso:.4f}")


RMSE validación Ridge: 11081.4106
R2 validación Ridge: 0.6003
RMSE validación Lasso: 11081.6468
R2 validación Lasso: 0.6003


### Decision tree


In [7]:
from sklearn.tree import DecisionTreeRegressor

# Barrido hiperparámetros
max_depths = [3, 5, 7, 9, 11, 13, 15, 17, 19, 21]
best_rmse = float('inf')
best_depth = None
for depth in max_depths:
    dt_model = DecisionTreeRegressor(max_depth=depth, random_state=42)
    dt_model.fit(x_train, y_train)
    y_pred_dt = dt_model.predict(x_val)
    rmse_dt = np.sqrt(mean_squared_error(y_val, y_pred_dt))
    
    if rmse_dt < best_rmse:
        best_rmse = rmse_dt
        best_depth = depth
print(f"Mejor RMSE: {best_rmse:.4f} con max_depth={best_depth}")

# Entrenar modelo con el mejor max_depth
best_dt_model = DecisionTreeRegressor(max_depth=best_depth, random_state=42)
best_dt_model.fit(x_train, y_train)
# Predecir con validación
y_pred_best_dt = best_dt_model.predict(x_val)
# Comparar contra los valores reales de validación
rmse_best_dt = np.sqrt(mean_squared_error(y_val, y_pred_best_dt))
print(f"RMSE validación Decision Tree (mejor max_depth={best_depth}): {rmse_best_dt:.4f}")
r2_val_best_dt = r2_score(y_val, y_pred_best_dt)
print(f"R2 validación Decision Tree (mejor max_depth={best_depth}): {r2_val_best_dt:.4f}")

#meto al mejor modelo en mlflow
with mlflow.start_run(run_name="Decision Tree Regressor"):
    # Log de métricas
    mlflow.log_metric("rmse_val", rmse_best_dt)
    mlflow.log_metric("r2_val", r2_val_best_dt)

    # Log de hiperparámetros
    mlflow.log_param("model_type", "DecisionTreeRegressor")
    mlflow.log_param("max_depth", best_depth)

    # Guardar modelo
    mlflow.sklearn.log_model(best_dt_model, "modelo_dt")

2025/07/02 17:52:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Mejor RMSE: 5990.7468 con max_depth=21
RMSE validación Decision Tree (mejor max_depth=21): 5990.7468
R2 validación Decision Tree (mejor max_depth=21): 0.8832


2025/07/02 17:53:01 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Decision Tree Regressor at: http://127.0.0.1:5000/#/experiments/454211106375854603/runs/ce20f70b07d74adc8723b12734e59707
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/454211106375854603


### Random Forest

In [8]:
from sklearn.ensemble import RandomForestRegressor

# barrido hiperparámetros random forest
n_estimators = [10, 50, 100, 200, 300]
max_depths_rf = [3, 5, 7, 9, 11]
best_rmse_rf = float('inf')
best_n_estimators = None
best_max_depth_rf = None
for n in n_estimators:
    for depth in max_depths_rf:
        rf_model = RandomForestRegressor(n_estimators=n, max_depth=depth, random_state=42)
        rf_model.fit(x_train, y_train)
        y_pred_rf = rf_model.predict(x_val)
        rmse_rf = np.sqrt(mean_squared_error(y_val, y_pred_rf))

        
        if rmse_rf < best_rmse_rf:
            best_rmse_rf = rmse_rf
            best_n_estimators = n
            best_max_depth_rf = depth
print(f"Mejor RMSE Random Forest: {best_rmse_rf:.4f} con n_estimators={best_n_estimators} y max_depth={best_max_depth_rf}")
# Entrenar modelo con los mejores hiperparámetros
best_rf_model = RandomForestRegressor(n_estimators=best_n_estimators, max_depth=best_max_depth_rf, random_state=42)
best_rf_model.fit(x_train, y_train)
# Predecir con validación
y_pred_best_rf = best_rf_model.predict(x_val)
# Comparar contra los valores reales de validación
rmse_best_rf = np.sqrt(mean_squared_error(y_val, y_pred_best_rf))
print(f"RMSE validación Random Forest (mejores hiperparámetros): {rmse_best_rf:.4f}")
r2_val_best_rf = r2_score(y_val, y_pred_best_rf)
print(f"R2 validación Random Forest (mejores hiperparámetros): {r2_val_best_rf:.4f}")

# meto al mejor modelo en mlflow
with mlflow.start_run(run_name="Random Forest Regressor"):
    # Log de métricas
    mlflow.log_metric("rmse_val", rmse_best_rf)
    mlflow.log_metric("r2_val", r2_val_best_rf)

    # Log de hiperparámetros
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("n_estimators", best_n_estimators)
    mlflow.log_param("max_depth", best_max_depth_rf)

    # Guardar modelo
    mlflow.sklearn.log_model(best_rf_model, "modelo_rf")


Mejor RMSE Random Forest: 5427.9166 con n_estimators=50 y max_depth=11


2025/07/02 17:53:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


RMSE validación Random Forest (mejores hiperparámetros): 5427.9166
R2 validación Random Forest (mejores hiperparámetros): 0.9041


2025/07/02 17:53:30 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Random Forest Regressor at: http://127.0.0.1:5000/#/experiments/454211106375854603/runs/466fc2a863744825ad39ec5a6e7a2fab
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/454211106375854603


### XGBOOST

In [9]:
try:
    from xgboost import XGBRegressor
except ImportError:
    !pip install xgboost


In [10]:
from xgboost import XGBRegressor
model = XGBRegressor()
# Entrenar modelo XGBoost
model.fit(x_train, y_train)
# Predecir con validación
y_pred_xgb = model.predict(x_val)
# Comparar contra los valores reales de validación
rmse_xgb = np.sqrt(mean_squared_error(y_val, y_pred_xgb))
print(f"RMSE validación XGBoost: {rmse_xgb:.4f}")
r2_val_xgb = r2_score(y_val, y_pred_xgb)
print(f"R2 validación XGBoost: {r2_val_xgb:.4f}")


RMSE validación XGBoost: 5124.8349
R2 validación XGBoost: 0.9145


In [11]:
#barrido hiperparámetros XGboost

param_grid = {
    'n_estimators': [50, 100, 200,300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'gamma': [0, 0.1, 0.2],
}
best_rmse_xgb = float('inf')
best_params_xgb = None
for n_estimators in param_grid['n_estimators']:
    for max_depth in param_grid['max_depth']:
        for learning_rate in param_grid['learning_rate']:
            for subsample in param_grid['subsample']:
                for gamma in param_grid['gamma']:
                    xgb_model = XGBRegressor(n_estimators=n_estimators, max_depth=max_depth,
                                             learning_rate=learning_rate, subsample=subsample, gamma=gamma)
                    xgb_model.fit(x_train, y_train)
                    y_pred_xgb = xgb_model.predict(x_val)
                    rmse_xgb = np.sqrt(mean_squared_error(y_val, y_pred_xgb))
                    
                    if rmse_xgb < best_rmse_xgb:
                        best_rmse_xgb = rmse_xgb
                        best_params_xgb = (n_estimators, max_depth, learning_rate, subsample, gamma)
print(f"Mejor RMSE XGBoost: {best_rmse_xgb:.4f} con parámetros: {best_params_xgb}")


#meto al mejor modelo en mlflow
with mlflow.start_run(run_name="XGBoost Regressor"):    
    #entreno con los mejores hiperparámetros
    best_xgb_model = XGBRegressor(n_estimators=best_params_xgb[0], 
                                   max_depth=best_params_xgb[1],
                                   learning_rate=best_params_xgb[2],
                                   subsample=best_params_xgb[3],
                                   gamma=best_params_xgb[4])
    best_xgb_model.fit(x_train, y_train)
    # Predecir con validación
    y_pred_best_xgb = best_xgb_model.predict(x_val)
    # Comparar contra los valores reales de validación
    rmse_best_xgb = np.sqrt(mean_squared_error(y_val, y_pred_best_xgb))
    print(f"RMSE validación XGBoost (mejores hiperparámetros): {rmse_best_xgb:.4f}")
    r2_val_best_xgb = r2_score(y_val, y_pred_best_xgb)
    print(f"R2 validación XGBoost (mejores hiperparámetros): {r2_val_best_xgb:.4f}")
    # Log de métricas
    mlflow.log_metric("rmse_val", rmse_best_xgb)
    mlflow.log_metric("r2_val", r2_val_best_xgb)
    # Log de hiperparámetros
    mlflow.log_param("model_type", "XGBoostRegressor")
    mlflow.log_param("n_estimators", best_params_xgb[0])
    mlflow.log_param("max_depth", best_params_xgb[1])
    mlflow.log_param("learning_rate", best_params_xgb[2])
    mlflow.log_param("subsample", best_params_xgb[3])
    mlflow.log_param("gamma", best_params_xgb[4])
    # Guardar modelo
    mlflow.sklearn.log_model(best_xgb_model, "modelo_xgb")


KeyboardInterrupt: 

### Prueba red neuronal

In [ ]:
# red neuronal
from sklearn.neural_network import MLPRegressor

# Barrido hiperparámetros MLP
hidden_layer_sizes = [(50,), (100,), (50, 50), (100, 50)]
activation_functions = ['relu']
best_rmse_mlp = float('inf')
best_params_mlp = None
for hidden_layer in hidden_layer_sizes:
    for activation in activation_functions:
        mlp_model = MLPRegressor(hidden_layer_sizes=hidden_layer, activation=activation, max_iter=10000, random_state=42, early_stopping=True)
        mlp_model.fit(x_train, y_train)
        y_pred_mlp = mlp_model.predict(x_val)
        rmse_mlp = np.sqrt(mean_squared_error(y_val, y_pred_mlp))
        
        if rmse_mlp < best_rmse_mlp:
            best_rmse_mlp = rmse_mlp
            best_params_mlp = (hidden_layer, activation)
print(f"Mejor RMSE MLP: {best_rmse_mlp:.4f} con parámetros: {best_params_mlp}")
# Entrenar modelo MLP con los mejores hiperparámetros
best_mlp_model = MLPRegressor(hidden_layer_sizes=best_params_mlp[0], 
                               activation=best_params_mlp[1],
                               max_iter=1000, random_state=42,
                               early_stopping=True)
best_mlp_model.fit(x_train, y_train)
# Predecir con validación
y_pred_best_mlp = best_mlp_model.predict(x_val)
# Comparar contra los valores reales de validación
rmse_best_mlp = np.sqrt(mean_squared_error(y_val, y_pred_best_mlp))
print(f"RMSE validación MLP (mejores hiperparámetros): {rmse_best_mlp:.4f}")
r2_val_best_mlp = r2_score(y_val, y_pred_best_mlp)
print(f"R2 validación MLP (mejores hiperparámetros): {r2_val_best_mlp:.4f}")
# meto al mejor modelo en mlflow
with mlflow.start_run(run_name="MLP Regressor"):
    # Log de métricas
    mlflow.log_metric("rmse_val", rmse_best_mlp)
    mlflow.log_metric("r2_val", r2_val_best_mlp)

    # Log de hiperparámetros
    mlflow.log_param("model_type", "MLPRegressor")
    mlflow.log_param("hidden_layer_sizes", best_params_mlp[0])
    mlflow.log_param("activation", best_params_mlp[1])

    # Guardar modelo
    mlflow.sklearn.log_model(best_mlp_model, "modelo_mlp")




Mejor RMSE MLP: 5935.3913 con parámetros: ((100, 50), 'relu')


/Users/kekozim/Library/Python/3.9/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
2025/07/02 02:20:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


RMSE validación MLP (mejores hiperparámetros): 5997.3394
R2 validación MLP (mejores hiperparámetros): 0.8714


2025/07/02 02:20:21 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run MLP Regressor at: http://127.0.0.1:5000/#/experiments/454211106375854603/runs/7037f2c2d5834c3f8aa8933de6d35aed
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/454211106375854603


### Pruebo mejor XGBoost con test


In [ ]:
# pruebo mejor XGB en test
x_test = test.copy().drop(columns=['Precio'])
y_test = test['Precio']
# Escalado
x_test = scaler.transform(x_test)   
# Predecir con test
y_pred_test_xgb = best_xgb_model.predict(x_test)
# Comparar contra los valores reales de test
rmse_test_xgb = np.sqrt(mean_squared_error(y_test, y_pred_test_xgb))
print(f"RMSE test XGBoost (mejores hiperparámetros): {rmse_test_xgb:.4f}")
r2_test_xgb = r2_score(y_test, y_pred_test_xgb)
print(f"R2 test XGBoost (mejores hiperparámetros): {r2_test_xgb:.4f}")


RMSE test XGBoost (mejores hiperparámetros): 3936.4802
R2 test XGBoost (mejores hiperparámetros): 0.9471
